In [ ]:
#Step 1
%pip install h3 geopandas
dbutils.library.restartPython()

In [ ]:
#Step 2
import os, h3, numpy as np, pandas as pd, geopandas as gpd
from sklearn.neighbors import BallTree

RAW = "/Volumes/workspace/trash/raw"
R = 6_371_000                      # Earth radius in meters
CITIES = {
    "nyc": {
        "type_col": "type",
        "categories": ["Dirty Condition", "Illegal Dumping",
                       "Litter Basket Request", "Litter Basket Complaint"],
        "years": 1,
    },
    "chicago": {
        "type_col": "type",
        "categories": ["Fly Dumping Complaint", "Sanitation Code Violation",
                       "Street Cleaning Request", "Wire Basket Request"],
        "years": 1,
    },
    "orlando": {"type_col": None, "categories": None, "years": 1},
}
PLACE_COLS = ["restaurants", "bars", "cafes", "transit_stops", "parks"]
CAT = {"restaurant": "restaurants", "fast_food": "restaurants", "bar": "bars", "pub": "bars",
       "cafe": "cafes", "bus_stop": "transit_stops", "station": "transit_stops", "park": "parks"}

def to_cell(df):
    return [h3.latlng_to_cell(a, b, 9) for a, b in zip(df.lat, df.lon)]

def tree_of(df):
    return BallTree(np.radians(df[["lat", "lon"]].values), metric="haversine")

def load_points(path):
    """Read a CSV and standardize column names, including lat/lon."""
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    for want, alts in {"lat": ["latitude", "y"], "lon": ["longitude", "lng", "long", "x"]}.items():
        if want not in df.columns:
            alt = next((a for a in alts if a in df.columns), None)
            if alt is None:
                raise ValueError(f"{path}: no {want} column. Columns are: {list(df.columns)}")
            df = df.rename(columns={alt: want})
    df["lat"] = pd.to_numeric(df.lat, errors="coerce")
    df["lon"] = pd.to_numeric(df.lon, errors="coerce")
    return df.dropna(subset=["lat", "lon"])

# OSM files usually split tags into columns like amenity / leisure / highway / railway
TAG_COLS = ["tag", "category", "kind", "amenity", "leisure", "highway", "railway", "public_transport"]

def derive_tag(p):
    present = [c for c in TAG_COLS if c in p.columns]
    if not present:
        raise ValueError(f"No tag-like column found. Columns are: {list(p.columns)}")
    return p[present].bfill(axis=1).iloc[:, 0]   # first non-empty tag value in each row

In [ ]:
#Step 3
def make_cells(slug):
    boundary = gpd.read_file(f"{RAW}/{slug}/boundary.geojson").to_crs(4326).union_all()
    c = pd.DataFrame({"h3": sorted(h3.geo_to_cells(boundary, 9))})
    c[["lat", "lon"]] = [h3.cell_to_latlng(x) for x in c.h3]
    c["city"] = slug
    return c

In [ ]:
#Step 4
def add_place_counts(c, slug):
    p = load_points(f"{RAW}/{slug}/osm_places.csv")
    p["cat"] = derive_tag(p).map(CAT)
    p = p.dropna(subset=["cat"])
    p["h3"] = to_cell(p)
    counts = p.groupby(["h3", "cat"]).size().unstack(fill_value=0).reindex(columns=PLACE_COLS, fill_value=0)
    c = c.merge(counts, left_on="h3", right_index=True, how="left")
    c[PLACE_COLS] = c[PLACE_COLS].fillna(0)
    return c

In [ ]:
#Step 5
def add_bin_features(c, slug):
    bins = load_points(f"{RAW}/{slug}/osm_bins.csv")
    t, X = tree_of(bins), np.radians(c[["lat", "lon"]].values)
    c["bins_200m"] = t.query_radius(X, r=200 / R, count_only=True)
    c["dist_nearest_bin_m"] = t.query(X, k=1)[0][:, 0] * R
    return c, bins

In [ ]:
#Step 6
def add_reports(c, slug, cfg):
    path = next((f"{RAW}/{slug}/{f}" for f in ["complaints.csv", "reports.csv"]
                 if os.path.exists(f"{RAW}/{slug}/{f}")), None)
    r = load_points(path) if path else pd.DataFrame(columns=["lat", "lon"])

    if cfg.get("categories") and len(r):
        col = cfg["type_col"]
        if col not in r.columns:
            raise ValueError(f"{slug}: no '{col}' column. Columns are: {list(r.columns)}")
        r = r[r[col].isin(cfg["categories"])]

    r["h3"] = to_cell(r)
    c = c.merge(r.groupby("h3").size().rename("reports"), left_on="h3", right_index=True, how="left")
    c["reports"] = c.reports.fillna(0)
    c["reports_per_yr"] = c.reports / cfg["years"]
    return c, r

In [ ]:
#Step 6 for every city
frames, bins_by_city, reports_by_city = [], {}, {}
for slug, cfg in CITIES.items():
    c = add_place_counts(make_cells(slug), slug)
    c, bins_by_city[slug] = add_bin_features(c, slug)
    c, reports_by_city[slug] = add_reports(c, slug, cfg)
    frames.append(c)
cells = pd.concat(frames, ignore_index=True)

# Target: complaint rate relative to the city's own average, so NYC and Chicago are comparable
cells["y"] = cells.reports_per_yr / cells.groupby("city").reports_per_yr.transform("mean")

spark.createDataFrame(cells).write.mode("overwrite").saveAsTable("workspace.trash.silver_cells")

In [ ]:
#Steps 1-6 Test
display(cells.groupby("city")[["reports", "bins_200m"] + PLACE_COLS].sum())

In [ ]:
#Step 7
import mlflow
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
mlflow.sklearn.autolog()

FEATURES = PLACE_COLS + ["bins_200m", "dist_nearest_bin_m"]
nyc, chi = cells[cells.city == "nyc"], cells[cells.city == "chicago"]

def train(df, name, test=None):
    with mlflow.start_run(run_name=name):
        m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05).fit(df[FEATURES], df.y)
        if test is not None:
            p = m.predict(test[FEATURES])
            mlflow.log_metrics({"xcity_r2": r2_score(test.y, p),
                                "xcity_spearman": spearmanr(test.y, p).statistic})
    return m

train(nyc, "train_nyc_test_chicago", test=chi)
train(chi, "train_chicago_test_nyc", test=nyc)
final = train(pd.concat([nyc, chi]), "final_nyc_chicago")

cells["pred"] = final.predict(cells[FEATURES]).clip(min=0)

In [ ]:
#Step 8
def norm(s):
    return (s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s * 0

g = cells.copy()
g["traffic"] = g[["transit_stops", "restaurants", "bars", "cafes"]].sum(axis=1)
g["traffic"] = g.groupby("city").traffic.transform(norm)            # 0–1 within each city
g["demand"] = g.pred * g.traffic

# Under-reported: the model expects a lot, but few reports come in
g["report_ratio"] = g.reports / g.pred.clip(lower=1e-6)
hi_pred = g.groupby("city").pred.transform(lambda s: s.quantile(0.75))
lo_ratio = g.groupby("city").report_ratio.transform(lambda s: s.quantile(0.25))
g["under_reported"] = (g.pred >= hi_pred) & (g.report_ratio <= lo_ratio)

spark.createDataFrame(g).write.mode("overwrite").saveAsTable("workspace.trash.cell_scores")

In [ ]:
#Step 9
def gain(c):  # PLACEHOLDER: demand, discounted when a bin is already within 200 m
    return c.demand * np.clip(c.nearest_bin_m / 200, 0, 1)

def pick_sites(city_cells, bins, reports, n=50):
    pts = bins[["lat", "lon"]].values.tolist()
    cand, sites = city_cells.copy(), []
    for rank in range(1, n + 1):
        t = BallTree(np.radians(pts), metric="haversine")
        cand["nearest_bin_m"] = t.query(np.radians(cand[["lat", "lon"]].values), k=1)[0][:, 0] * R
        cand["gain"] = gain(cand)
        best = cand.loc[cand.gain.idxmax()]
        sites.append({"rank": rank, "h3": best.h3, "lat": best.lat, "lon": best.lon,
                      "gain": best.gain, "nearest_bin_m": best.nearest_bin_m})
        pts.append([best.lat, best.lon])
        cand = cand.drop(best.name)
    s = pd.DataFrame(sites)
    s["nearby_reports"] = (tree_of(reports).query_radius(np.radians(s[["lat", "lon"]].values),
                           r=200 / R, count_only=True) if len(reports) else 0)
    return s

sites_by_city = {slug: pick_sites(g[g.city == slug], bins_by_city[slug], reports_by_city[slug])
                 for slug in CITIES}
sites = pd.concat([s.assign(city=k) for k, s in sites_by_city.items()], ignore_index=True)
spark.createDataFrame(sites).write.mode("overwrite").saveAsTable("workspace.trash.bin_candidates")

In [ ]:
#Step 10
COLUMNS = {
    "cell_scores": ["h3_cell", "lat", "lng", "expected_overflow", "reports_per_1k",
                    "under_reported", "demand"],
    "bins": ["lat", "lng", "source"],
    "bin_candidates": ["rank", "lat", "lng", "gain", "nearest_bin_m", "nearby_reports"],
}

g = (spark.table("workspace.trash.cell_scores").toPandas()
       .rename(columns={"h3": "h3_cell", "lon": "lng", "pred": "expected_overflow"}))
g["reports_per_1k"] = np.nan          # no population data any more → loads as NULL

sites = spark.table("workspace.trash.bin_candidates").toPandas().rename(columns={"lon": "lng"})

for slug in CITIES:
    out = f"{RAW}/gold/{slug}"
    os.makedirs(out, exist_ok=True)
    g[g.city == slug][COLUMNS["cell_scores"]].to_csv(f"{out}/cell_scores.csv", index=False)
    sites[sites.city == slug][COLUMNS["bin_candidates"]].to_csv(f"{out}/bin_candidates.csv", index=False)
    bins = load_points(f"{RAW}/{slug}/osm_bins.csv").rename(columns={"lon": "lng"}).assign(source="osm")
    bins[COLUMNS["bins"]].to_csv(f"{out}/bins.csv", index=False)
    print(f"{slug}: {(g.city == slug).sum():,} cells, {(sites.city == slug).sum()} sites, {len(bins):,} bins")